In [4]:
import json
import shutil
from pathlib import Path
import random

# ---------------- CONFIG ----------------
IMAGES_DIR = Path("images/train")
ANNOTATIONS_DIR = Path("annotations/train")

OUTPUT_DIR = Path("dataset")
TRAIN_RATIO = 0.8
RANDOM_SEED = 42
# ----------------------------------------

random.seed(RANDOM_SEED)

def convert_bbox(bbox, img_w, img_h):
    """
    Convert Mapillary bbox to YOLO format.
    """
    xmin = bbox["xmin"]
    ymin = bbox["ymin"]
    xmax = bbox["xmax"]
    ymax = bbox["ymax"]

    x_center = ((xmin + xmax) / 2) / img_w
    y_center = ((ymin + ymax) / 2) / img_h
    width = (xmax - xmin) / img_w
    height = (ymax - ymin) / img_h

    return x_center, y_center, width, height


def main():
    image_files = sorted(IMAGES_DIR.glob("*"))
    image_files = [p for p in image_files if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]

    random.shuffle(image_files)
    split_idx = int(len(image_files) * TRAIN_RATIO)

    splits = {
        "train": image_files[:split_idx],
        "val": image_files[split_idx:]
    }

    for split in splits:
        (OUTPUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (OUTPUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

    for split, files in splits.items():
        for img_path in files:
            ann_path = ANNOTATIONS_DIR / f"{img_path.stem}.json"

            if not ann_path.exists():
                print(f"⚠️ Missing annotation for {img_path.name}")
                continue

            with open(ann_path, "r") as f:
                data = json.load(f)

            img_w = data["width"]
            img_h = data["height"]

            label_lines = []

            for obj in data.get("objects", []):
                bbox = obj.get("bbox")
                if bbox is None:
                    continue

                x, y, w, h = convert_bbox(bbox, img_w, img_h)

                # single class: streetsign = 0
                label_lines.append(f"0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

            # Write label file
            label_path = OUTPUT_DIR / "labels" / split / f"{img_path.stem}.txt"
            with open(label_path, "w") as f:
                f.write("\n".join(label_lines))

            # Copy image
            shutil.copy(img_path, OUTPUT_DIR / "images" / split / img_path.name)

    print("✅ Conversion completed")


if __name__ == "__main__":
    main()


✅ Conversion completed


In [5]:
from pathlib import Path
import random

# ---------------- CONFIG ----------------
TRAIN_IMAGES_DIR = Path("dataset/images/train")
TRAIN_LABELS_DIR = Path("dataset/labels/train")

NUM_IMAGES_TO_KEEP = 900
RANDOM_SEED = 42
# ----------------------------------------

random.seed(RANDOM_SEED)


def main():
    # Get all training images
    image_files = sorted([
        p for p in TRAIN_IMAGES_DIR.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ])

    if len(image_files) <= NUM_IMAGES_TO_KEEP:
        print(f"Nothing to delete: only {len(image_files)} images found.")
        return

    # Randomly select images to keep
    images_to_keep = set(random.sample(image_files, NUM_IMAGES_TO_KEEP))

    deleted_images = 0
    deleted_labels = 0

    for img_path in image_files:
        if img_path not in images_to_keep:
            # Delete image
            img_path.unlink()
            deleted_images += 1

            # Delete corresponding label file
            label_path = TRAIN_LABELS_DIR / f"{img_path.stem}.txt"
            if label_path.exists():
                label_path.unlink()
                deleted_labels += 1

    print("✅ Done")
    print(f"Kept images: {NUM_IMAGES_TO_KEEP}")
    print(f"Deleted images: {deleted_images}")
    print(f"Deleted labels: {deleted_labels}")


if __name__ == "__main__":
    main()

✅ Done
Kept images: 900
Deleted images: 8857
Deleted labels: 8857
